Rating_Prediction

In [141]:
import pandas as pd
from modules.preprocessing import preprocess_text
df = pd.read_csv("datasets/amazon_reviews.csv")

In [142]:
df

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,day_diff,helpful_yes,total_vote
0,A3SBTW3WS4IQSN,B007WTAJTO,NaN,"[0, 0]",No issues.,4.0,Four Stars,1406073600,2014-07-23,138,0,0
1,A18K1ODH1I2MVB,B007WTAJTO,0mie,"[0, 0]","Purchased this for my device, it worked as adv...",5.0,MOAR SPACE!!!,1382659200,2013-10-25,409,0,0
2,A2FII3I2MBMUIA,B007WTAJTO,1K3,"[0, 0]",it works as expected. I should have sprung for...,4.0,nothing to really say....,1356220800,2012-12-23,715,0,0
3,A3H99DFEG68SR,B007WTAJTO,1m2,"[0, 0]",This think has worked out great.Had a diff. br...,5.0,Great buy at this price!!! *** UPDATE,1384992000,2013-11-21,382,0,0
4,A375ZM4U047O79,B007WTAJTO,2&amp;1/2Men,"[0, 0]","Bought it with Retail Packaging, arrived legit...",5.0,best deal around,1373673600,2013-07-13,513,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
4910,A2LBMKXRM5H2W9,B007WTAJTO,"ZM ""J""","[0, 0]",I bought this Sandisk 16GB Class 10 to use wit...,1.0,Do not waste your money.,1374537600,2013-07-23,503,0,0
4911,ALGDLRUI1ZPCS,B007WTAJTO,Zo,"[0, 0]",Used this for extending the capabilities of my...,5.0,Great item!,1377129600,2013-08-22,473,0,0
4912,A2MR1NI0ENW2AD,B007WTAJTO,Z S Liske,"[0, 0]",Great card that is very fast and reliable. It ...,5.0,Fast and reliable memory card,1396224000,2014-03-31,252,0,0
4913,A37E6P3DSO9QJD,B007WTAJTO,Z Taylor,"[0, 0]",Good amount of space for the stuff I want to d...,5.0,Great little card,1379289600,2013-09-16,448,0,0


In [143]:
df = df[["reviewText", "overall"]]

In [144]:
df

,reviewText,overall
0,No issues.,4.0
1,"Purchased this for my device, it worked as adv...",5.0
2,it works as expected. I should have sprung for...,4.0
3,This think has worked out great.Had a diff. br...,5.0
4,"Bought it with Retail Packaging, arrived legit...",5.0
...,...,...
4910,I bought this Sandisk 16GB Class 10 to use wit...,1.0
4911,Used this for extending the capabilities of my...,5.0
4912,Great card that is very fast and reliable. It ...,5.0
4913,Good amount of space for the stuff I want to d...,5.0


In [145]:
df.rename(
    columns={
        "reviewText": "review",
        "overall": "rating"
    },
    inplace=True
)

In [146]:
df.dropna(inplace=True)

In [147]:
print(df["rating"].value_counts().sort_index())

rating
1.0     244
2.0      80
3.0     142
4.0     527
5.0    3921
Name: count, dtype: int64


In [148]:
df["clean_review"] = df["review"].apply(preprocess_text)

In [149]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)
X = vectorizer.fit_transform(df["clean_review"])

y = df["rating"].astype(int)

In [150]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [151]:
from sklearn.linear_model import LogisticRegression


In [152]:
param_grid = {
    "C": [0.1, 0.5, 1, 2, 5, 10],
    "solver": ["lbfgs", "saga"],
    "class_weight": [None, "balanced"],
    "max_iter": [1000, 2000]
}

In [153]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    LogisticRegression(),
    param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

In [154]:
print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

Best Parameters: {'C': 10, 'class_weight': 'balanced', 'max_iter': 1000, 'solver': 'lbfgs'}
Best CV Score: 0.763955620651217


In [155]:
y_pred = best_model.predict(X_test)

In [156]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7741607324516785
              precision    recall  f1-score   support

           1       0.52      0.64      0.57        44
           2       0.00      0.00      0.00        12
           3       0.27      0.10      0.15        30
           4       0.21      0.19      0.20       107
           5       0.87      0.90      0.88       790

    accuracy                           0.77       983
   macro avg       0.37      0.36      0.36       983
weighted avg       0.75      0.77      0.76       983

[[ 28   1   2   4   9]
 [  8   0   1   1   2]
 [  5   1   3   3  18]
 [  4   0   2  20  81]
 [  9   0   3  68 710]]


In [157]:
import pandas as pd

results = pd.DataFrame({
    "Actual Rating": y_test.values,
    "Predicted Rating": y_pred.round(1)
})

results.head(20)

,Actual Rating,Predicted Rating
0,5,5
1,1,1
2,5,5
3,5,5
4,4,5
5,5,5
6,4,5
7,5,5
8,5,5
9,5,5


In [158]:
review = "Worst product I have ever purchased."

clean_review = preprocess_text(review)

review_vector = vectorizer.transform([clean_review])

prediction = best_model.predict(review_vector)[0]

print("Predicted Rating:", prediction)

Predicted Rating: 5


In [160]:
import joblib

joblib.dump(best_model, "models/rating_model.pkl")
joblib.dump(vectorizer, "models/rating_vectorizer.pkl")

['models/rating_vectorizer.pkl']